# Lab 10 · Auditoría de sesgos y riesgos del modelo telco

Notebook para auditar un modelo de churn: accuracy global, métricas por grupo, falsos positivos/negativos, paridad demográfica, igualdad de oportunidades, variables proxy y gobernanza.

In [ ]:
# Instalación de dependencias básicas. En SageMaker normalmente ya están instaladas.
%pip -q install pandas numpy matplotlib networkx scikit-learn s3fs

import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Por defecto se leen los CSV incluidos en la carpeta local data/.
# Si prefieres leer desde S3, cambia USE_S3=True y ajusta S3_PREFIX.
USE_S3 = False
DATA_DIR = Path('data')
S3_PREFIX = 's3://TU_BUCKET/TU_CARPETA'  # ejemplo: s3://mi-bucket/graph

def data_path(filename):
    return f"{S3_PREFIX}/{filename}" if USE_S3 else str(DATA_DIR / filename)

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

from sklearn.metrics import confusion_matrix

## Parte A. Cargar datos de auditoría

In [ ]:
df = pd.read_csv(data_path('telco_model_audit_lab10.csv'))
print(df.shape)
display(df.head())

## Parte B. Rendimiento global y desagregado

In [ ]:
accuracy_global = (df['y_true'] == df['y_pred']).mean()
print(f'Accuracy global: {accuracy_global:.3f}')

acc_region = df.groupby('region').apply(lambda g: (g['y_true'] == g['y_pred']).mean()).rename('accuracy').sort_values()
acc_zone = df.groupby('zone_type').apply(lambda g: (g['y_true'] == g['y_pred']).mean()).rename('accuracy').sort_values()
display(acc_region.round(3).reset_index())
display(acc_zone.round(3).reset_index())
acc_region.to_csv(OUTPUT_DIR/'lab10_accuracy_region.csv')
acc_zone.to_csv(OUTPUT_DIR/'lab10_accuracy_zone_type.csv')

In [ ]:
plt.figure(figsize=(8,5))
acc_region.sort_values().plot(kind='bar')
plt.ylim(0,1)
plt.ylabel('accuracy')
plt.title('Accuracy por región')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'lab10_accuracy_por_region.png', dpi=150, bbox_inches='tight')
plt.show()

## Parte C. Disparidad en errores

In [ ]:
def metricas_grupo(g):
    tp = int(((g.y_true==1) & (g.y_pred==1)).sum())
    tn = int(((g.y_true==0) & (g.y_pred==0)).sum())
    fp = int(((g.y_true==0) & (g.y_pred==1)).sum())
    fn = int(((g.y_true==1) & (g.y_pred==0)).sum())
    precision = tp/(tp+fp) if (tp+fp)>0 else np.nan
    recall = tp/(tp+fn) if (tp+fn)>0 else np.nan
    fpr = fp/(fp+tn) if (fp+tn)>0 else np.nan
    return pd.Series({'clientes':len(g), 'TP':tp, 'TN':tn, 'FP':fp, 'FN':fn, 'precision':precision, 'recall':recall, 'FPR':fpr})

met_region = df.groupby('region').apply(metricas_grupo).sort_values('recall')
display(met_region.round(3))
met_region.to_csv(OUTPUT_DIR/'lab10_metricas_error_region.csv')

## Parte D. Métricas formales de equidad

In [ ]:
# Paridad demográfica: tasa de predicciones positivas por grupo.
paridad = df.groupby('region')['y_pred'].mean().rename('tasa_positivos').sort_values(ascending=False)
# Igualdad de oportunidades: recall por grupo.
recall = met_region['recall']
print('Paridad demográfica por región:')
display(paridad.round(3).reset_index())
print('Brecha de tasa de positivos:', round(paridad.max() - paridad.min(), 3))
print('Recall por región:')
display(recall.round(3).reset_index())
print('Brecha de recall:', round(recall.max() - recall.min(), 3))

pd.concat([paridad, recall.rename('recall')], axis=1).to_csv(OUTPUT_DIR/'lab10_metricas_equidad_region.csv')

## Parte E. Variables proxy y bucle de retroalimentación

In [ ]:
proxy_zone = df.groupby('zone_type').agg(
    clientes=('customer_id','count'),
    churn_real=('y_true','mean'),
    churn_predicho=('y_pred','mean'),
    precio_medio=('monthly_price','mean'),
    tenure_medio=('tenure_months','mean')
).round(3).sort_values('churn_predicho', ascending=False)
display(proxy_zone)
proxy_zone.to_csv(OUTPUT_DIR/'lab10_proxy_zone_type.csv')

print('Reflexión: si zone_type o region resumen desigualdad territorial, pueden actuar como proxy. No siempre deben eliminarse, pero sí documentarse y auditarse.')

## Parte F. Veredicto y mitigación

Completa esta celda con un veredicto razonado. La estructura siguiente te sirve para el entregable.

In [ ]:
mitigacion = pd.DataFrame([
    {'medida':'Recalibrar umbrales por grupo', 'objetivo':'reducir brecha de recall sin disparar falsos positivos'},
    {'medida':'Reentrenar con datos más equilibrados', 'objetivo':'mejorar rendimiento en regiones con peor accuracy'},
    {'medida':'Monitorización continua', 'objetivo':'detectar deriva de rendimiento y equidad'},
    {'medida':'Model card y revisión humana', 'objetivo':'documentar limitaciones y evitar decisiones automáticas dañinas'},
])
display(mitigacion)
mitigacion.to_csv(OUTPUT_DIR/'lab10_mitigacion_gobernanza.csv', index=False)

## Cierre

Un modelo con buena accuracy global puede ser inaceptable si falla sistemáticamente para determinados grupos. El entregable debe terminar con un veredicto: desplegar, desplegar con condiciones o no desplegar.